In [1]:
# importing libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from xgboost import XGBClassifier

train_df = pd.read_pickle("../data/train.pkl")

In [2]:
from sklearn.metrics import (
    accuracy_score ,
    precision_score ,
    recall_score ,
    f1_score ,
    average_precision_score , 
    roc_auc_score ,
    confusion_matrix ,
    classification_report, 
    ConfusionMatrixDisplay 
)

In [3]:
train_df = train_df.sort_values(["Transaction_Day" , "Transaction_Time_Of_Day"]).reset_index(drop=True)

In [4]:
x = train_df.drop(columns=["isFraud" , "TransactionID" ,"TransactionDT"])
y = train_df["isFraud"]

In [5]:
# NUMERICAL

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median' , add_indicator=True)) ,
    ('scaler' , StandardScaler())
])

In [6]:
# CATEGORICAL

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant' , fill_value="Missing")) ,
    ('encoder' , OneHotEncoder(handle_unknown='ignore'))
])

In [7]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5 , gap=0)

In [8]:
seeds = [123 , 2024 , 777]
xgb_seed_results = {}

for seed in seeds:
    print(f"\n{'='*60}")
    print(f"XGBoost - SEED{seed}")
    print(f"{'='*60}")

    oof_predictions = np.full(len(x) , np.nan)

    for fold , (train_idx , val_idx) in enumerate(tscv.split(x) , start=1):
        x_train = x.iloc[train_idx].copy()
        x_val = x.iloc[val_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_val = y.iloc[val_idx].copy()

        numeric_cols = x_train.select_dtypes(include=["int64" , "float64" , "uint16" , "uint8"]).columns
        categorical_cols = x_train.select_dtypes(include=["object"]).columns

        # Combined Pipeline
        preprocessor = ColumnTransformer([
            ('num' , numeric_pipeline , numeric_cols) ,
            ('cat' , categorical_pipeline , categorical_cols)
        ])

        xgb_pipeline = Pipeline([
            ('preprocessor' , preprocessor) ,
            ('model' , XGBClassifier(random_state=seed , n_jobs=-1 , max_depth=10 , n_estimators=100 , subsample=0.8,colsample_bytree=0.8))
        ])
        
        xgb_pipeline.fit(x_train , y_train)
        val_pred = xgb_pipeline.predict(x_val)
        val_prob = xgb_pipeline.predict_proba(x_val)[:,1]
        oof_predictions[val_idx] = val_prob

        roc_auc = roc_auc_score(y_val , val_prob)
        pr_auc = average_precision_score(y_val , val_prob)
        recall = recall_score(y_val , val_pred)
        f1 = f1_score(y_val , val_pred)
        precision = precision_score(y_val , val_pred)


        print(
            f"Fold {fold} | "
            f"ROC-AUC : {roc_auc:.4f} | " 
            f"PR-AUC : {pr_auc:.4f} | "
            f"Recall = {recall:.4f} | "
            f"F1 Score = {f1:.4f} | "
            f"Precision = {precision:.4f}")
        print(confusion_matrix(y_val , val_pred))

    oof_mask = ~np.isnan(oof_predictions)
    overall_roc_auc = roc_auc_score(y.iloc[oof_mask] , oof_predictions[oof_mask])
    overall_pr_auc = average_precision_score(y.iloc[oof_mask] , oof_predictions[oof_mask])
    print(f"OOF ROC-AUC : {overall_roc_auc:.4f}")
    print(f"OOF PR-AUC : {overall_pr_auc:.4f}")
    xgb_seed_results[seed] = {
        "oof" : oof_predictions ,
        "roc-auc" : overall_roc_auc , 
        "pr-auc" : overall_pr_auc
    }

    pd.DataFrame({
        "isFraud" : y.values , 
        f"xgboost_seed_{seed}_oof" : xgb_seed_results[seed]["oof"]
    }).to_csv(f"../outputs/oof/xgboost_seed_{seed}_oof.csv" , index=False)


XGBoost - SEED123


Fold 1 | ROC-AUC : 0.8624 | PR-AUC : 0.4856 | Recall = 0.3036 | F1 Score = 0.4444 | Precision = 0.8291
[[94909   207]
 [ 2303  1004]]
Fold 2 | ROC-AUC : 0.8873 | PR-AUC : 0.5583 | Recall = 0.4019 | F1 Score = 0.5312 | Precision = 0.7830
[[93913   452]
 [ 2427  1631]]
Fold 3 | ROC-AUC : 0.8856 | PR-AUC : 0.5073 | Recall = 0.3185 | F1 Score = 0.4586 | Precision = 0.8187
[[94477   260]
 [ 2512  1174]]
Fold 4 | ROC-AUC : 0.9078 | PR-AUC : 0.5546 | Recall = 0.4197 | F1 Score = 0.5312 | Precision = 0.7236
[[94146   591]
 [ 2139  1547]]
Fold 5 | ROC-AUC : 0.9127 | PR-AUC : 0.5455 | Recall = 0.3859 | F1 Score = 0.5044 | Precision = 0.7279
[[94536   490]
 [ 2086  1311]]
OOF ROC-AUC : 0.8759
OOF PR-AUC : 0.5235

XGBoost - SEED2024
Fold 1 | ROC-AUC : 0.8728 | PR-AUC : 0.4966 | Recall = 0.2894 | F1 Score = 0.4301 | Precision = 0.8373
[[94930   186]
 [ 2350   957]]
Fold 2 | ROC-AUC : 0.8907 | PR-AUC : 0.5668 | Recall = 0.4110 | F1 Score = 0.5351 | Precision = 0.7665
[[93857   508]
 [ 2390  1668]]
F